# Capítulo 2 · Llamadas a funciones a fondo

La llamada a funciones, capa por capa, con el agente de la Clínica Veterinaria Naranco: el texto exacto que recibe el modelo, los tokens especiales de sus llamadas, dos analizadores, varias llamadas por paso y cómo obligar al modelo a usar una herramienta escribiendo el principio de su respuesta. Se usan dos modelos, Qwen3-4B-Instruct-2507 y Qwen3-1.7B.

**Antes de empezar, activa la GPU:** menú *Entorno de ejecución* → *Cambiar tipo de entorno de ejecución* → *GPU T4* → *Guardar*.

La explicación completa está en el capítulo en PDF. Este cuaderno es el laboratorio.

## 1. Cargar el modelo y la clínica

La clínica y sus herramientas son las del juego 4 del capítulo 1, con una más: `precio`.

In [ ]:
!pip install -q transformers==5.19.0  # las versiones con las que se comprobó el libro

In [ ]:
"""Capítulo 2: llamadas a funciones a fondo. Lo que entra en el modelo, lo que sale,
cómo se leen las llamadas, varias llamadas por paso y forzar una herramienta."""
import copy
import json
import re
import time
import urllib.request
from datetime import date
from pathlib import Path

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

MODELO_GPU = "Qwen/Qwen3-4B-Instruct-2507"  # con la GPU T4 gratuita de Colab
MODELO_CPU = "Qwen/Qwen3-1.7B"              # sin GPU: más pequeño, y peor agente
MODELO = None                               # None: elegir según el entorno
DATOS = Path("datos")
REPOSITORIO = "https://raw.githubusercontent.com/HernanDiaz/ai-tools-mcp-skills/main/"
HOY = date(2026, 10, 20)                    # la fecha de los datos de la clínica


def cargar(nombre=None):
    """Carga el modelo y su tokenizador: en la GPU, en float16; si no, en la CPU."""
    gpu = torch.cuda.is_available()
    if nombre is None:
        nombre = MODELO_GPU if gpu else MODELO_CPU
    tok = AutoTokenizer.from_pretrained(nombre)
    modelo = AutoModelForCausalLM.from_pretrained(
        nombre, dtype=torch.float16 if gpu else torch.float32)
    modelo.to("cuda" if gpu else "cpu").eval()
    print(f"Modelo {nombre} en {'cuda' if gpu else 'cpu'}")
    return tok, modelo


def descargar(archivo):
    """Un archivo JSON de la carpeta datos; en Colab, se descarga."""
    ruta = DATOS / archivo
    if not ruta.exists():
        DATOS.mkdir(exist_ok=True)
        urllib.request.urlretrieve(REPOSITORIO + ruta.as_posix(), ruta)
    return json.loads(ruta.read_text(encoding="utf-8"))

In [ ]:
tok, modelo = cargar(MODELO)

In [ ]:
def reiniciar():
    """Deja la clínica como al principio: la agenda original y ningún aviso."""
    global CITAS, AVISOS
    CITAS, AVISOS = copy.deepcopy(CITAS0), []


def sin_tildes(texto):
    """Minúsculas y sin tildes, para comparar nombres escritos de cualquier forma."""
    return str(texto).lower().translate(str.maketrans("áéíóúü", "aeiouu")).strip()


def reservar_cita(mascota, veterinario, fecha, hora, motivo):
    """Reserva un hueco libre de la agenda; devuelve la cita o el problema."""
    if mascota not in MASCOTAS:
        return {"error": f"No existe la mascota {mascota}."}
    if veterinario not in VETERINARIOS:
        return {"error": f"No existe el veterinario {veterinario}."}
    if any(c["fecha"] == fecha and c["hora"] == hora and c["veterinario"] == veterinario
           and c["estado"] != "cancelada" for c in CITAS):
        return {"error": f"{VETERINARIOS[veterinario]} ya tiene una cita el {fecha} "
                         f"a las {hora}."}
    cita = {"id": f"CT-{len(CITAS0) + len(CITAS) + 1:04d}", "fecha": fecha, "hora": hora,
            "veterinario": veterinario, "mascota": mascota, "motivo": motivo,
            "estado": "pendiente"}
    CITAS.append(cita)
    return cita


def cancelar_cita(cita):
    """Cancela una cita pendiente."""
    for c in CITAS:
        if c["id"] == cita:
            if c["estado"] != "pendiente":
                return {"error": f"La cita {cita} está «{c['estado']}»."}
            c["estado"] = "cancelada"
            return {"cancelada": cita, "fecha": c["fecha"], "hora": c["hora"]}
    return {"error": f"No existe la cita {cita}."}


def avisar(dueno, texto):
    """Envía un aviso al dueño (aquí se guarda en una lista)."""
    if dueno not in DUENOS:
        return {"error": f"No existe el cliente {dueno}."}
    AVISOS.append({"dueno": dueno, "texto": texto})
    return {"enviado": True, "a": DUENOS[dueno]["nombre"],
            "por": DUENOS[dueno]["avisos"]}


def edad(nacimiento):
    """Años cumplidos el día de los datos."""
    n = date.fromisoformat(nacimiento)
    return HOY.year - n.year - ((HOY.month, HOY.day) < (n.month, n.day))


def resumen_mascota(mid):
    """Lo esencial de una mascota, sin el historial completo."""
    m = MASCOTAS[mid]
    return {"id": mid, "nombre": m["nombre"], "especie": m["especie"], "raza": m["raza"],
            "edad_anos": edad(m["nacimiento"]), "peso_actual_kg": m["peso_kg"],
            "alergias": m["alergias"] or "ninguna conocida",
            "dueno": {"id": m["dueno"], "nombre": DUENOS[m["dueno"]]["nombre"]}}


def vacunas_con_estado(mid):
    """Cada vacuna con su próxima fecha y si ya está vencida."""
    return [{"vacuna": v["vacuna"], "proxima": v["proxima"],
             "estado": "vencida" if v["proxima"] < HOY.isoformat() else "al día"}
            for v in VACUNAS[mid]]


def ficha_cliente_3(id_cliente):
    if id_cliente not in DUENOS:
        return {"error": f"No existe el cliente {id_cliente}. Búscalo por nombre con "
                         "buscar(tabla='duenos', campo='nombre', valor=...)."}
    pendientes = [f for f in FACTURAS if f["dueno"] == id_cliente and not f["pagada"]]
    return {"id": id_cliente, "nombre": DUENOS[id_cliente]["nombre"],
            "avisos_por": DUENOS[id_cliente]["avisos"],
            "mascotas": [{"id": m, "nombre": d["nombre"], "especie": d["especie"]}
                         for m, d in MASCOTAS.items() if d["dueno"] == id_cliente],
            "facturas_sin_pagar": [{"id": f["id"], "fecha": f["fecha"], "total": f["total"]}
                                   for f in pendientes],
            "total_sin_pagar": round(sum(f["total"] for f in pendientes), 2)}


def agenda_3(fecha):
    if not re.fullmatch(r"\d{4}-\d{2}-\d{2}", str(fecha)):
        return {"error": f"La fecha «{fecha}» no es válida; usa AAAA-MM-DD."}
    citas = [c for c in CITAS if c["fecha"] == fecha and c["estado"] != "cancelada"]
    return {"fecha": fecha, "citas": [f"{c['id']} {c['hora']} {c['veterinario']} "
                                      f"{MASCOTAS[c['mascota']]['nombre']} ({c['mascota']})"
                                      for c in citas],
            "veterinarios": VETERINARIOS,
            "horas_libres": {v: [h for h in HORAS if not any(
                c["hora"] == h and c["veterinario"] == v for c in citas)]
                for v in VETERINARIOS}}


def localizar(mascota, dueno=""):
    """El id de la mascota a partir de su nombre (y el del dueño, si hay varias)."""
    ids = [m for m, d in MASCOTAS.items() if sin_tildes(mascota) == sin_tildes(d["nombre"])
           and sin_tildes(dueno) in sin_tildes(DUENOS[d["dueno"]]["nombre"])]
    if not ids:
        return None, {"error": f"No hay ninguna mascota llamada «{mascota}»"
                               + (f" de {dueno}" if dueno else "") + " en la clínica."}
    if len(ids) > 1:
        return None, {"error": f"Hay {len(ids)} mascotas llamadas «{mascota}»; indica el "
                               "nombre del dueño.",
                      "mascotas": [resumen_mascota(m) for m in ids]}
    return ids[0], None


def veterinario_de(nombre):
    """El id del veterinario a partir de su nombre o de su id."""
    for v, n in VETERINARIOS.items():
        if sin_tildes(nombre) in (sin_tildes(v), sin_tildes(n)) or \
                sin_tildes(nombre) in sin_tildes(n):
            return v
    return None


def buscar_mascota(nombre, dueno=""):
    mid, error = localizar(nombre, dueno)
    return error or resumen_mascota(mid)


def mascotas_de(dueno):
    ids = [d for d, x in DUENOS.items() if sin_tildes(dueno) in sin_tildes(x["nombre"])]
    if not ids:
        return {"error": f"No hay ningún cliente llamado «{dueno}»."}
    return {"clientes": [{"nombre": DUENOS[d]["nombre"], "mascotas": [
        {"nombre": x["nombre"], "especie": x["especie"], "raza": x["raza"]}
        for m, x in MASCOTAS.items() if x["dueno"] == d]} for d in ids]}


def vacunas(mascota, dueno=""):
    mid, error = localizar(mascota, dueno)
    return error or {"mascota": MASCOTAS[mid]["nombre"], "hoy": HOY.isoformat(),
                     "vacunas": vacunas_con_estado(mid)}


def historial(mascota, ultimas=3, dueno=""):
    mid, error = localizar(mascota, dueno)
    if error:
        return error
    return {"mascota": MASCOTAS[mid]["nombre"], "visitas": [
        {k: v[k] for k in ("fecha", "motivo", "tratamiento", "peso_kg")}
        for v in HISTORIAL[mid][-int(ultimas):]]}


def dosis(medicamento, mascota, dueno=""):
    med = FARMACIA.get(sin_tildes(medicamento))
    if med is None:
        return {"error": f"No tenemos «{medicamento}». Medicamentos: "
                         + ", ".join(FARMACIA) + "."}
    mid, error = localizar(mascota, dueno)
    if error:
        return error
    m = MASCOTAS[mid]
    avisos = []
    if sin_tildes(medicamento) in [sin_tildes(a) for a in m["alergias"]]:
        avisos.append(f"{m['nombre']} es alérgico a {medicamento}: no recetar.")
    if m["especie"] not in med["especies"]:
        avisos.append(f"{medicamento} no está indicado en {m['especie']}s.")
    return {"medicamento": sin_tildes(medicamento), "mascota": m["nombre"],
            "peso_kg": m["peso_kg"], "dosis_mg_por_kg": med["dosis_mg_kg"],
            "dosis_mg_por_toma": round(med["dosis_mg_kg"] * m["peso_kg"], 2),
            "existencias": med["existencias"],
            "reposicion": med.get("reposicion", "sin fecha") if not med["existencias"] else None,
            "avisos": avisos or "ninguno"}


def huecos_libres(fecha, veterinario=""):
    r = agenda_3(fecha)
    if "error" in r:
        return r
    vets = [veterinario_de(veterinario)] if veterinario else list(VETERINARIOS)
    if vets == [None]:
        return {"error": f"No hay ningún veterinario «{veterinario}». Son: "
                         + ", ".join(VETERINARIOS.values()) + "."}
    return {"fecha": fecha, "horas_libres": {VETERINARIOS[v]: r["horas_libres"][v]
                                             for v in vets}}


def citas_de(mascota, dueno=""):
    mid, error = localizar(mascota, dueno)
    return error or {"mascota": MASCOTAS[mid]["nombre"], "hoy": HOY.isoformat(), "citas": [
        {"id": c["id"], "fecha": c["fecha"], "hora": c["hora"], "motivo": c["motivo"],
         "veterinario": VETERINARIOS[c["veterinario"]], "estado": c["estado"]}
        for c in CITAS if c["mascota"] == mid]}


def cuentas(dueno):
    ids = [d for d, x in DUENOS.items() if sin_tildes(dueno) in sin_tildes(x["nombre"])]
    if not ids:
        return {"error": f"No hay ningún cliente llamado «{dueno}»."}
    return {"clientes": [{k: v for k, v in ficha_cliente_3(d).items() if k != "id"}
                         for d in ids]}


def reservar(mascota, veterinario, fecha, hora, motivo, dueno=""):
    mid, error = localizar(mascota, dueno)
    if error:
        return error
    vet = veterinario_de(veterinario)
    if vet is None:
        return {"error": f"No hay ningún veterinario «{veterinario}». Son: "
                         + ", ".join(VETERINARIOS.values()) + "."}
    r = reservar_cita(mid, vet, fecha, hora, motivo)
    if "error" in r:
        return r
    return {"reservada": r["id"], "mascota": MASCOTAS[mid]["nombre"],
            "veterinario": VETERINARIOS[vet], "fecha": fecha, "hora": hora}


def avisar_dueno_de(mascota, texto, dueno=""):
    mid, error = localizar(mascota, dueno)
    return error or avisar(MASCOTAS[mid]["dueno"], texto)


def palabras(texto):
    """Las raíces (siete primeras letras) de las palabras con contenido de un texto."""
    return {p[:7] for p in re.findall(r"\w{4,}", sin_tildes(texto))}


def precio(servicio):
    """Precio de un servicio de la clínica o de un medicamento de la farmacia: los que
    comparten más palabras con lo que se pide."""
    precios = {**TARIFAS, **{k: d["precio"] for k, d in FARMACIA.items()}}
    comunes = {k: len(palabras(servicio) & palabras(k)) for k in precios}
    mejor = max(comunes.values())
    if mejor == 0:
        return {"error": f"No hay ninguna tarifa de «{servicio}». Servicios: "
                         + ", ".join(TARIFAS) + "; y los medicamentos de la farmacia."}
    return {"precios_euros": {k: precios[k] for k, c in comunes.items() if c == mejor}}


FUNCIONES = {"buscar_mascota": buscar_mascota, "mascotas_de": mascotas_de,
             "vacunas": vacunas, "historial": historial, "dosis": dosis,
             "huecos_libres": huecos_libres, "citas_de": citas_de, "cuentas": cuentas,
             "reservar_cita": reservar, "cancelar_cita": cancelar_cita,
             "avisar_dueno": avisar_dueno_de, "precio": precio}

In [ ]:
def herramienta(nombre, descripcion, parametros):
    """Esquema de una herramienta; los parámetros sin valor por defecto son obligatorios."""
    return {"type": "function", "function": {
        "name": nombre, "description": descripcion,
        "parameters": {"type": "object",
                       "required": [p for p, (_, oblig) in parametros.items() if oblig],
                       "properties": {p: {"type": "string", "description": d}
                                      for p, (d, _) in parametros.items()}}}}


MASCOTA = ("Nombre de la mascota.", True)


DUENO = ("Nombre del dueño; solo si hay varias mascotas con ese nombre.", False)


ESQUEMAS_TAREAS = [
    herramienta("buscar_mascota", "Datos de una mascota buscada por su nombre: especie, "
                "edad, peso actual, alergias y dueño.", {"nombre": MASCOTA, "dueno": DUENO}),
    herramienta("mascotas_de", "Las mascotas de un cliente, buscado por su nombre.",
                {"dueno": ("Nombre del dueño.", True)}),
    herramienta("vacunas", "Vacunas de una mascota: la próxima fecha de cada una y si "
                "está vencida.", {"mascota": MASCOTA, "dueno": DUENO}),
    herramienta("historial", "Las últimas visitas de una mascota: fecha, motivo, "
                "tratamiento y peso.", {"mascota": MASCOTA,
                                        "ultimas": ("Cuántas visitas (por defecto, 3).", False),
                                        "dueno": DUENO}),
    herramienta("dosis", "Dosis de un medicamento para una mascota según su peso actual, "
                "con avisos de alergia o especie y las existencias.",
                {"medicamento": ("Nombre del medicamento.", True), "mascota": MASCOTA,
                 "dueno": DUENO}),
    herramienta("huecos_libres", "Horas libres de la agenda en un día, de un veterinario o "
                "de todos.", {"fecha": ("AAAA-MM-DD.", True),
                              "veterinario": ("Nombre del veterinario; vacío para todos.", False)}),
    herramienta("citas_de", "Citas de una mascota (pasadas y pendientes), con su id.",
                {"mascota": MASCOTA, "dueno": DUENO}),
    herramienta("cuentas", "Facturas sin pagar de un cliente, buscado por su nombre, y el "
                "total que debe.", {"dueno": ("Nombre del dueño.", True)}),
    herramienta("reservar_cita", "Reserva una cita en una hora libre.",
                {"mascota": MASCOTA, "veterinario": ("Nombre del veterinario.", True),
                 "fecha": ("AAAA-MM-DD.", True), "hora": ("HH:MM, en punto o y media.", True),
                 "motivo": ("Motivo de la cita.", True), "dueno": DUENO}),
    herramienta("cancelar_cita", "Cancela una cita pendiente.",
                {"cita": ("Id de la cita, como CT-0026 (lo da citas_de).", True)}),
    herramienta("avisar_dueno", "Envía un aviso al dueño de una mascota, por correo o SMS "
                "según prefiera.", {"mascota": MASCOTA, "texto": ("Texto del aviso, en "
                                                                   "español.", True),
                                     "dueno": DUENO}),
]


ESQUEMAS = ESQUEMAS_TAREAS + [
    herramienta("precio", "Precio en euros de un servicio de la clínica (consulta, vacunas, "
                "análisis…) o de un medicamento.", {"servicio": ("Servicio o medicamento.", True)})]


BASE = (
    "You are the assistant of the staff of Clínica Veterinaria Naranco, a veterinary "
    "clinic in Oviedo (Spain). Today is Tuesday 2026-10-20. Do the task step by step "
    "using the tools. Never invent data that a tool can give you, and never say that "
    "an action was done unless a tool confirmed it. ")


FINAL = "When the task is finished, answer in Spanish, in one or two sentences."


SISTEMA_UNA = BASE + (
    "Call only one tool at a time, and before each tool call write one short line "
    "starting with 'Thought:' that says what you already know and what you need next. ") + FINAL


SISTEMA_VARIAS = BASE + (
    "Before calling tools, write one short line starting with 'Thought:' that says what "
    "you already know and what you need next. If several calls do not depend on each "
    "other, request them all in the same turn. ") + FINAL

## 2. Capa 1: lo que entra

El texto exacto que recibe el modelo, y cuántos tokens ocupa cada parte.

In [ ]:
def plantilla(mensajes, esquemas):
    """El texto exacto que recibe el modelo, con las herramientas incluidas."""
    return tok.apply_chat_template(mensajes, tools=esquemas, tokenize=False,
                                   add_generation_prompt=True, enable_thinking=False)


def contar(texto):
    return len(tok(texto, add_special_tokens=False)["input_ids"])


def partes_de_la_entrada(tarea, sistema=SISTEMA_UNA, esquemas=ESQUEMAS):
    """Cuántos tokens ocupa cada parte del texto de entrada en el primer paso."""
    texto = plantilla([{"role": "system", "content": sistema},
                       {"role": "user", "content": tarea}], esquemas)
    inicio, fin = texto.rindex("<tools>"), texto.rindex("</tools>") + len("</tools>")
    herramientas = texto[inicio:fin]
    instrucciones = texto[texto.index("# Tools"):inicio] + texto[fin:texto.index("<|im_end|>")]
    return {"total": contar(texto), "sistema": contar(sistema), "tarea": contar(tarea),
            "herramientas": contar(herramientas), "instrucciones_de_llamada": contar(instrucciones)}

In [ ]:
def dice(respuesta, *formas):
    """¿Aparece en la respuesta alguna de esas formas?"""
    return any(re.search(f, respuesta or "", re.I) for f in formas)


def fecha_en(respuesta, iso):
    """¿Aparece la fecha, escrita como 2027-08-22, 22/08/2027 o 22 de agosto de 2027?"""
    a, m, d = iso.split("-")
    meses = ["enero", "febrero", "marzo", "abril", "mayo", "junio", "julio", "agosto",
             "septiembre", "octubre", "noviembre", "diciembre"]
    return dice(respuesta, iso, rf"\b0?{int(d)}/0?{int(m)}/{a}",
                rf"\b0?{int(d)} de {meses[int(m) - 1]}( de {a})?")


def citas_nuevas():
    return [c for c in CITAS if c["id"] not in {x["id"] for x in CITAS0}]


def primera_libre(fecha, vet):
    ocupadas = {c["hora"] for c in CITAS0 if c["fecha"] == fecha and c["veterinario"] == vet}
    return next(h for h in HORAS if h not in ocupadas)


def debe(nombre):
    return round(sum(f["total"] for f in FACTURAS if not f["pagada"]
                     and DUENOS[f["dueno"]]["nombre"] == nombre), 2)


def importe(respuesta, valor):
    """¿Aparece el importe, como 136, 136,00 o 136.00?"""
    entero = f"{valor:.2f}".replace(".00", "")
    return dice(respuesta, rf"\b{entero}([,.]00?)?\b", f"{valor:.2f}".replace(".", ","))


def aviso_a(nombre, *formas):
    return any(DUENOS[a["dueno"]]["nombre"] == nombre
               and all(re.search(f, a["texto"], re.I) for f in formas) for a in AVISOS)


TAREAS = [
    ("vacuna_rabia", "¿Cuándo le toca a Lola, la perra de Marta Fernández, la próxima "
     "vacuna de la rabia?",
     lambda r: fecha_en(r, "2027-08-22")),
    ("dosis", "¿Cuántos miligramos de meloxicam le corresponden a Max en cada toma?",
     lambda r: dice(r, r"2[,.]07", r"2[,.]1\b")),
    ("alergia", "Van a recetarle amoxicilina a Simba. ¿Hay algún problema?",
     lambda r: dice(r, r"al[eé]rg")),
    ("reservar", "Reserva para Toby una revisión con el Dr. Pablo Menéndez el jueves 22, "
     "en la primera hora libre.",
     lambda r: [(c["mascota"], c["veterinario"], c["fecha"], c["hora"]) for c in citas_nuevas()]
     == [("M-002", "V2", "2026-10-22", primera_libre("2026-10-22", "V2"))]),
    ("cancelar", "Cancela la cita que tiene Rocky mañana.",
     lambda r: [c["id"] for c in CITAS if c["estado"] == "cancelada"] == ["CT-0060"]),
    ("vencidas_aviso", "Avisa al dueño de Toby de las vacunas que tiene vencidas.",
     lambda r: aviso_a("Pedro Álvarez", "rabia", "polivalente", "leishmaniosis")),
    ("deuda", "¿Cuánto debe Óscar Prieto en facturas sin pagar?",
     lambda r: importe(r, debe("Óscar Prieto"))),
    ("gata_de_marta", "Marta Fernández llama por su gata. ¿Tiene alguna vacuna vencida?",
     lambda r: dice(r, "trivalente") and dice(r, "vencid|caducad|atrasad|pasad")),
    ("peso", "¿Cuánto pesaba Zeus en su última visita, y cuánto ha cambiado respecto a "
     "la anterior?",
     lambda r: dice(r, r"32[,.]2") and dice(r, r"0[,.]6")
     and not dice(r, r"últim[ao] visita[^.,]{0,30}(2025|31[,.]6)", r"31[,.]6 kg en (su|la) últim")),
    ("ultima_visita", "¿Por qué vino Nube la última vez y qué tratamiento le pusieron?",
     lambda r: dice(r, "herida") and dice(r, "amoxicilina")),
    ("huecos", "¿Qué horas tiene libres la Dra. Ana Suárez el miércoles 21 por la tarde?",
     lambda r: all(dice(r, h) for h in ("16:00", "16:30", "17:30", "18:00", "18:30"))
     and not dice(r, r"17:00")),
    ("no_existe", "¿Cuándo le toca la vacuna de la rabia a Pancho?",
     lambda r: dice(r, r"no (hay|existe|encuentr|figura|tenemos|aparece)", "ning[uú]n")
     and not dice(r, r"\b20\d\d-\d\d-\d\d\b")),
]


def cargar_datos():
    global MASCOTAS, DUENOS, HISTORIAL, VACUNAS, CITAS0, FARMACIA, FACTURAS
    global VETERINARIOS, HORAS, TARIFAS, RESPUESTAS
    MASCOTAS, DUENOS = descargar("mascotas.json"), descargar("duenos.json")
    HISTORIAL, VACUNAS = descargar("historial.json"), descargar("vacunas.json")
    CITAS0, FARMACIA = descargar("citas.json"), descargar("farmacia.json")
    FACTURAS, VETERINARIOS = descargar("facturas.json"), descargar("veterinarios.json")
    TARIFAS = descargar("tarifas.json")
    HORAS = [f"{h:02d}:{m:02d}" for h in range(9, 14) for m in (0, 30)] + \
            [f"{h:02d}:{m:02d}" for h in range(16, 19) for m in (0, 30)]
    RESPUESTAS = []
    reiniciar()

In [ ]:
cargar_datos()
tarea = "¿Cuánto deben en total, entre los tres, Óscar Prieto, Paula Iglesias y Laura Gómez?"
texto = plantilla([{"role": "system", "content": SISTEMA_UNA}, {"role": "user", "content": tarea}],
                  ESQUEMAS[:2])
print(texto)
print(partes_de_la_entrada(tarea))

## 3. Capa 2: lo que sale

`<tool_call>` es un solo token del vocabulario.

In [ ]:
print(tok("<tool_call>", add_special_tokens=False)["input_ids"])

In [ ]:
def generar(mensajes, esquemas, prefijo="", max_tokens=400):
    """Una respuesta del modelo. Con prefijo, el texto de la respuesta empieza por él."""
    texto = plantilla(mensajes, esquemas) + prefijo
    entrada = tok(texto, return_tensors="pt", add_special_tokens=False).to(modelo.device)
    with torch.no_grad():
        salida = modelo.generate(**entrada, max_new_tokens=max_tokens,
                                 do_sample=False, temperature=None, top_p=None,
                                 top_k=None, pad_token_id=tok.eos_token_id)
    nuevos = salida[0, entrada["input_ids"].shape[1]:]
    respuesta = prefijo + tok.decode(nuevos, skip_special_tokens=True)
    return respuesta.strip(), entrada["input_ids"].shape[1], len(nuevos)

In [ ]:
respuesta, entrada, salida = generar([{"role": "system", "content": SISTEMA_VARIAS},
                                      {"role": "user", "content": tarea}], ESQUEMAS)
print(entrada, "tokens de entrada,", salida, "de salida")
print(respuesta)

## 4. Capa 3: leer las llamadas

In [ ]:
def leer_ingenuo(respuesta):
    """El analizador más sencillo: el JSON que hay tras la primera <tool_call>."""
    if "<tool_call>" not in respuesta:
        return []
    bloque = respuesta.split("<tool_call>")[1].split("</tool_call>")[0]
    try:
        llamada = json.loads(bloque)
        return [{"name": llamada["name"], "arguments": llamada.get("arguments", {})}]
    except (ValueError, KeyError, TypeError):
        return [{"name": None, "arguments": {}}]


def leer_llamadas(respuesta):
    """Todas las llamadas, cada una entre <tool_call> y </tool_call> (el del capítulo 1)."""
    llamadas = []
    for bloque in re.findall(r"<tool_call>(.*?)</tool_call>", respuesta, re.S):
        try:
            llamada = json.loads(bloque)
            llamadas.append({"name": llamada["name"],
                             "arguments": llamada.get("arguments", {})})
        except (ValueError, KeyError, TypeError):
            llamadas.append({"name": None, "arguments": {}})
    return llamadas


ANALIZADORES = {"ingenuo": leer_ingenuo, "completo": leer_llamadas}

In [ ]:
for nombre, leer in ANALIZADORES.items():
    print(nombre, leer(respuesta))

## 5. El agente, con varias llamadas por paso y con la primera forzada

In [ ]:
def ejecutar(llamada):
    """Ejecuta una llamada si la herramienta existe y los argumentos encajan."""
    funcion = FUNCIONES.get(llamada["name"])
    if funcion is None:
        return {"error": f"No existe la herramienta {llamada['name']}."}
    try:
        return funcion(**llamada["arguments"])
    except (TypeError, ValueError, KeyError) as e:
        return {"error": f"Argumentos no válidos: {e}"}


def agente(tarea, sistema=SISTEMA_UNA, varias=False, forzar=False, leer=leer_llamadas,
           max_pasos=8, ver=False):
    """El bucle del capítulo 1. Con varias, ejecuta todas las llamadas de un paso; con
    forzar, el primer paso empieza por <tool_call> y tiene que llamar a una herramienta.
    Guarda cada respuesta tal cual en RESPUESTAS, para probar después los analizadores."""
    mensajes = [{"role": "system", "content": sistema}, {"role": "user", "content": tarea}]
    traza = []
    for paso in range(1, max_pasos + 1):
        prefijo = "<tool_call>\n" if forzar and paso == 1 else ""
        respuesta, entrada, salida = generar(mensajes, ESQUEMAS, prefijo)
        RESPUESTAS.append(respuesta)
        llamadas = leer(respuesta)
        if not varias:
            llamadas = llamadas[:1]
        pensamiento = respuesta.split("<tool_call>")[0].strip()
        traza.append({"paso": paso, "tokens_entrada": entrada, "tokens_salida": salida,
                      "pensamiento": pensamiento, "llamadas": []})
        if not llamadas:
            if ver:
                print(f"[{paso}] Respuesta: {respuesta}")
            return respuesta, traza
        if ver and pensamiento:
            print(f"[{paso}] {pensamiento}")
        mensajes.append({"role": "assistant", "content": pensamiento, "tool_calls": [
            {"type": "function", "function": l} for l in llamadas if l["name"]]})
        for llamada in llamadas:
            resultado = ejecutar(llamada)
            traza[-1]["llamadas"].append({**llamada, "resultado": resultado})
            mensajes.append({"role": "tool", "name": llamada["name"] or "?",
                             "content": json.dumps(resultado, ensure_ascii=False)})
            if ver:
                print(f"[{paso}] {llamada['name']}({json.dumps(llamada['arguments'], ensure_ascii=False)})")
    return None, traza

In [ ]:
reiniciar()
agente(tarea, sistema=SISTEMA_VARIAS, varias=True, ver=True)

Las tareas paralelas, las de precios y las funciones que las resuelven y las miden:

In [ ]:
def vencidas(nombre):
    mid = next(m for m, d in MASCOTAS.items() if d["nombre"] == nombre)
    return [v["vacuna"] for v in vacunas_con_estado(mid) if v["estado"] == "vencida"]


def ultima_visita(nombre):
    mid = next(m for m, d in MASCOTAS.items() if d["nombre"] == nombre)
    return HISTORIAL[mid][-1]["fecha"]


def dosis_de(medicamento, nombre):
    m = next(d for d in MASCOTAS.values() if d["nombre"] == nombre)
    return FARMACIA[medicamento]["dosis_mg_kg"] * m["peso_kg"]


def cifra(respuesta, valor):
    """¿Aparece la cifra, con coma o con punto y con uno o dos decimales?"""
    return dice(respuesta, *{f"{valor:.2f}".replace(".", "[,.]"),
                             f"{valor:.1f}".replace(".", "[,.]") + r"\b",
                             rf"\b{valor:g}\b".replace(".", "[,.]")})


# Tareas con varias consultas independientes entre sí
PARALELAS = [
    ("deuda_de_tres", "¿Cuánto deben en total, entre los tres, Óscar Prieto, Paula Iglesias "
     "y Laura Gómez?",
     lambda r: importe(r, debe("Óscar Prieto") + debe("Paula Iglesias") + debe("Laura Gómez"))),
    ("dosis_de_tres", "¿Cuántos miligramos de meloxicam le corresponden por toma a Max, a Zeus "
     "y a Rufo?",
     lambda r: all(cifra(r, round(dosis_de("meloxicam", n), 2)) for n in ("Max", "Zeus", "Rufo"))),
    ("ultimas_visitas", "¿Cuándo vinieron por última vez Rocky, Nube y Tigre?",
     lambda r: all(fecha_en(r, ultima_visita(n)) for n in ("Rocky", "Nube", "Tigre"))),
    ("vencidas_de_tres", "¿Tienen alguna vacuna vencida Bruno, Nala o Salem? Dime cuáles.",
     lambda r: all(dice(r, v.split()[0]) for n in ("Bruno", "Nala", "Salem")
                   for v in vencidas(n))),
    ("mascotas_de_pedro", "¿Qué vacunas tienen vencidas las mascotas de Pedro Álvarez?",
     lambda r: all(dice(r, v) for v in vencidas("Toby") + vencidas("Copo"))),
    ("avisar_a_dos", "Avisa a los dueños de Oliver y de Salem de que tienen vencida la "
     "vacuna trivalente felina.",
     lambda r: aviso_a("Clara Miranda", "trivalente") and aviso_a("Rosa Villa", "trivalente")),
    ("huecos_de_dos", "¿Qué horas tienen libres el Dr. Pablo Menéndez y la Dra. Lucía Fano el "
     "viernes 23 por la mañana?",
     lambda r: all(dice(r, h) for v in ("V2", "V3") for h in libres_manana("2026-10-23", v))),
    ("precio_de_toby", "¿Cuánto costaría ponerle a Toby todas las vacunas que tiene vencidas?",
     lambda r: importe(r, sum(TARIFAS["vacuna " + v] for v in vencidas("Toby")))),
]


def libres_manana(fecha, vet):
    ocupadas = {c["hora"] for c in CITAS0 if c["fecha"] == fecha and c["veterinario"] == vet
                and c["estado"] != "cancelada"}
    return [h for h in HORAS if h < "14:00" and h not in ocupadas]


# Preguntas de precios, que el modelo podría contestar de memoria
PRECIOS = [
    ("limpieza", "¿Cuánto cuesta una limpieza dental?", lambda r: importe(r, 120)),
    ("trivalente", "¿Cuánto cobramos por la vacuna trivalente felina?", lambda r: importe(r, 36)),
    ("meloxicam", "¿Cuánto vale el meloxicam?", lambda r: cifra(r, 18.5)),
    ("analisis", "¿Qué precio tiene un análisis de sangre?", lambda r: importe(r, 65)),
    ("consulta", "¿Cuánto cuesta una consulta?", lambda r: importe(r, 35)),
    ("mixomatosis", "¿Cuánto cuesta vacunar a un conejo de la mixomatosis?",
     lambda r: importe(r, 25)),
    ("desparasitar", "Un cliente pregunta cuánto le costaría desparasitar a su perro.",
     lambda r: importe(r, 15)),
    ("doxiciclina", "¿Cuánto cuesta la doxiciclina y la tenemos ahora?",
     lambda r: cifra(r, 14.2) and dice(r, "no (hay|tenemos|queda)", "agotad", "sin existencias", r"\b0\b")),
]


def resolver(tareas, max_pasos=8, **opciones):
    """Resuelve cada tarea con la clínica reiniciada y anota el resultado."""
    filas = []
    for clave, tarea, comprobar in tareas:
        reiniciar()
        inicio = time.time()
        respuesta, traza = agente(tarea, max_pasos=max_pasos, **opciones)
        filas.append({"tarea": clave, "bien": bool(comprobar(respuesta)), "respuesta": respuesta,
                      "pasos": len(traza), "traza": traza,
                      "llamadas": [(l["name"], l["arguments"]) for p in traza for l in p["llamadas"]],
                      "tokens_entrada": sum(p["tokens_entrada"] for p in traza),
                      "segundos": round(time.time() - inicio, 1), "avisos": AVISOS})
    return filas


def resumen(filas):
    return {"bien": sum(f["bien"] for f in filas),
            "terminadas": sum(f["respuesta"] is not None for f in filas),
            "pasos": sum(f["pasos"] for f in filas),
            "llamadas": sum(len(f["llamadas"]) for f in filas),
            "sin_herramientas": sum(not f["llamadas"] for f in filas),
            "tokens_entrada": sum(f["tokens_entrada"] for f in filas),
            "segundos": round(sum(f["segundos"] for f in filas), 1)}


def contar_llamadas(respuestas):
    """Para cada analizador, las llamadas válidas que encuentra en las respuestas, frente a
    las que el modelo empezó a escribir (las etiquetas <tool_call>)."""
    escritas = sum(r.count("<tool_call>") for r in respuestas)
    return {"escritas": escritas, **{nombre: sum(
        sum(l["name"] in FUNCIONES and isinstance(l["arguments"], dict) for l in leer(r))
        for r in respuestas) for nombre, leer in ANALIZADORES.items()}}

In [ ]:
resultados = {}
for nombre, tareas, opciones in [
        ("Paralelas, una cada vez", PARALELAS, {}),
        ("Paralelas, varias por paso", PARALELAS, {"sistema": SISTEMA_VARIAS, "varias": True}),
        ("Paralelas, varias, analizador ingenuo", PARALELAS,
         {"sistema": SISTEMA_VARIAS, "varias": True, "leer": leer_ingenuo}),
        ("Precios", PRECIOS, {}),
        ("Precios, forzando una herramienta", PRECIOS, {"forzar": True})]:
    resultados[nombre] = resolver(tareas, **opciones)
    print(f"{nombre:40s}", resumen(resultados[nombre]))
print(contar_llamadas(RESPUESTAS))

## 6. El modelo pequeño

Qwen3-1.7B con el mismo programa. Primero se libera la memoria del modelo grande.

In [ ]:
del modelo
torch.cuda.empty_cache()
tok, modelo = cargar(MODELO_CPU)
reiniciar()
print(agente("¿Cuánto cuesta la vacuna trivalente felina?")[0])
reiniciar()
print(agente("¿Cuánto cuesta la vacuna trivalente felina?", forzar=True)[0])

In [ ]:
for nombre, opciones in [("Precios", {}), ("Precios, forzando una herramienta", {"forzar": True})]:
    print(f"{nombre:40s}", resumen(resolver(PRECIOS, **opciones)))

## Ejercicios

1. Explica con tus palabras qué hace la plantilla de chat. ¿Dónde acaban los esquemas de las herramientas? ¿Y los resultados?

2. Imprime el texto que genera `plantilla` para una tarea con solo dos herramientas y localiza las instrucciones fijas que añade la plantilla. ¿Cuántos tokens ocupan? ¿Cambian si cambias las herramientas?

3. Comprueba con el tokenizador que `<tool_call>` es un solo token. Haz lo mismo con `<tool_response>` y con `<tools>`. ¿Qué ventaja tiene que lo sean?

4. Escribe a mano cinco respuestas difíciles para los analizadores: dos llamadas pegadas sin salto de línea, una llamada con texto después, un JSON con comillas simples, una llamada sin cierre y una respuesta que menciona la etiqueta en una frase. ¿Qué devuelve cada analizador?

5. Ejecuta la tarea de la deuda de tres clientes con el analizador ingenuo y `ver=True`. ¿En qué paso pide el modelo las llamadas que se perdieron?

6. Escribe dos tareas en las que pedir varias llamadas a la vez sería un error, porque la segunda necesita el resultado de la primera. ¿Las pide a la vez el agente con `SISTEMA_VARIAS`?

7. Quita la línea «Thought:» de `SISTEMA_UNA` y prueba Qwen3-1.7B, sin forzar, con las doce tareas del capítulo 1. ¿Llama ahora a las herramientas?

8. Fuerza una herramienta concreta, no solo una llamada: el prefijo puede incluir el principio del JSON con el nombre. Úsalo para que las preguntas de precios empiecen siempre por `precio`. ¿Cambia algo con el modelo pequeño?

9. Prueba el modelo pequeño forzando todos los pasos, no solo el primero. ¿Qué pasa cuando ya tiene la respuesta y está obligado a llamar a una herramienta?

10. **Proyecto.** Añade a la clínica una herramienta `servicios()` sin argumentos, que devuelva la lista de servicios con su precio, y quita `precio`. Mide las preguntas de precios con los dos modelos, con y sin forzar, y compara los tokens de sus resultados con los de `precio`. ¿Qué diseño prefieres?

## Referencias

Modelos [Qwen3-4B-Instruct-2507](https://huggingface.co/Qwen/Qwen3-4B-Instruct-2507) y [Qwen3-1.7B](https://huggingface.co/Qwen/Qwen3-1.7B). Documentación de [herramientas y plantillas de chat](https://huggingface.co/docs/transformers/main/en/chat_extras) de Transformers y de [llamadas a funciones](https://qwen.readthedocs.io/en/latest/framework/function_call.html) de Qwen. Lecciones 13.02 y 13.03 de [AI Engineering from Scratch](https://github.com/rohitg00/ai-engineering-from-scratch). La Clínica Veterinaria Naranco y sus datos son ficticios. Código y experimentos propios.